# Cool Roof Priority Index — Mussafah, Abu Dhabi

**Arab Youth Space Hackathon 2026 · 813 Challenge**
**Theme:** Urban Expansion, Land Use Change & Heat Risk · **Track:** Hyperspectral
**Team:** T0050 · United Arab Emirates

---

### The problem this solves

A municipality knows its city is hot. What it does not know is **which specific roof
to treat first**, and **how much cooling that treatment actually buys**. Existing
urban heat island maps stop at diagnosis. This notebook turns open Earth
observation into a ranked, costed worklist of individual rooftops.

### Pipeline

| # | Step | Data | Output |
|---|------|------|--------|
| 3 | Heat and land cover, two epochs | Landsat C2 L2 (thermal + optical) | LST maps, ΔLST, built-up change |
| 4 | Validate the classification | ESA WorldCover 2021 | accuracy / precision / recall / F1 / IoU |
| 5 | Urban expansion and its thermal cost | the above | **°C penalty of desert→industrial conversion** |
| 6 | Surface brightness | Sentinel-2 L2A | broadband albedo, 20 m |
| 7 | **Calibrate the lever** | regression + spatial block CV | **°C of cooling per unit albedo, measured here** |
| 8 | Rank the roofs | OSM footprints + zonal stats | ranked CSV, GeoJSON |

### Two design decisions worth defending

**Landsat alone carries the change analysis.** Both epochs, both the thermal and the
optical indices, come from the same instrument family. Mixing Landsat 2014 with
Sentinel-2 2025 would fold a cross-sensor calibration difference into what we report
as real-world change. Sentinel-2 is used only for the present-day albedo layer,
where its finer resolution genuinely helps and no cross-epoch comparison is made.

**Level-2, not Level-1.** USGS Collection 2 Level-2 ships an atmospherically
corrected Surface Temperature product. We use it rather than hand-rolling a
mono-window algorithm from top-of-atmosphere radiance: fewer assumptions, a
documented and citable processing chain, and our effort goes where it is novel.

### Honest framing

Land surface temperature is not air temperature. These outputs are a **screening and
prioritisation indicator**, not a thermal model of the built environment and not
ground truth. Section 10 states precisely where this breaks.

**Runtime ≈ 25–40 minutes** on a free Colab CPU instance. No GPU.


## 1 · Environment

In [ ]:
%pip install -q \
    pystac-client==0.8.6 planetary-computer==1.0.0 stackstac==0.5.1 \
    rioxarray==0.17.0 rasterstats==0.20.0 scikit-learn==1.5.2 \
    geopandas==1.0.1 requests==2.32.4

In [ ]:
# ---------------------------------------------------------------------------
# Bootstrap: works whether this notebook was opened standalone in Colab, from a
# clone of the repo, or locally. Sets the working directory so relative paths
# resolve the same way everywhere.
# ---------------------------------------------------------------------------
import os, sys, subprocess
REPO_URL, REPO_DIR = "https://github.com/RISKYXBT/coolroof-mussafah.git", "coolroof-mussafah"

if not (os.path.isdir("../results") and os.path.isdir("../data")):
    if os.path.isdir(f"/content/{REPO_DIR}/notebooks"):
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    elif "google.colab" in sys.modules:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            r = subprocess.run(["git","clone","-q",REPO_URL,REPO_DIR],
                               capture_output=True, text=True)
            if r.returncode:
                print("clone failed:", r.stderr.strip())
                os.makedirs(f"{REPO_DIR}/notebooks", exist_ok=True)
        os.chdir(f"/content/{REPO_DIR}/notebooks")
    else:
        os.makedirs("../results", exist_ok=True)
        os.makedirs("../data/sample_input", exist_ok=True)

os.makedirs("../results", exist_ok=True)
os.makedirs("../data/sample_input", exist_ok=True)
print("working directory:", os.getcwd())

In [ ]:
import json, warnings, requests
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd
import matplotlib.pyplot as plt
import pystac_client, planetary_computer, stackstac, rioxarray
from shapely.geometry import box, Polygon
from rasterstats import zonal_stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")
np.random.seed(42)                     # reproducible CV fold assignment
plt.rcParams["figure.dpi"] = 110
RESULTS, SAMPLE = "../results", "../data/sample_input"
print("imports ok")

## 2 · Configuration

Every parameter the analysis depends on is in this one cell, so a reviewer can see
the whole experimental setup at a glance and switch city in one place.

In [ ]:
# ---- Area of interest -------------------------------------------------------
# Mussafah: Abu Dhabi's principal industrial district. Chosen because it has very
# large flat roofs (the cheapest cool-roof targets per m2), a strong thermal
# signal, and a decade of visible expansion into bare desert.
AOI_NAME   = "Mussafah, Abu Dhabi"
AOI_BBOX   = [54.43, 24.29, 54.58, 24.41]     # [W, S, E, N]
FOCUS_BBOX = [54.48, 24.33, 54.55, 24.38]     # smaller core for the per-roof step

# ---- Time windows -----------------------------------------------------------
EPOCH_A, EPOCH_A_NAME = "2014-05-01/2016-09-30", "2014-2016"
EPOCH_B, EPOCH_B_NAME = "2024-05-01/2026-09-30", "2024-2026"
S2_WINDOW             = "2025-05-01/2026-09-30"   # albedo only, present day

# ---- Limits -----------------------------------------------------------------
# Recon found 93 and 176 Landsat scenes. We take the cleanest N: a median over
# 40 cloud-screened scenes is already stable, and capping keeps Colab within
# memory and the runtime sensible.
MAX_CLOUD      = 15
MAX_SCENES_LS  = 40
MAX_SCENES_S2  = 20

EPSG, RES_LST, RES_S2 = 32640, 30, 20      # UTM 40N - correct zone for Abu Dhabi
TARGET_ALBEDO = 0.60                        # what a cool-roof coating reaches

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace)

print(f"AOI    : {AOI_NAME} {AOI_BBOX}")
print(f"epochs : {EPOCH_A_NAME} vs {EPOCH_B_NAME}")
print(f"caps   : <= {MAX_SCENES_LS} Landsat/epoch, <= {MAX_SCENES_S2} Sentinel-2")

## 3 · Landsat Collection 2 Level-2 — temperature *and* land cover, both epochs

One request per epoch pulls the thermal band and the optical bands together, so the
heat map and the land-cover map are derived from exactly the same scenes.

- Surface temperature: `lwir11`, scaled `K = DN × 0.00341802 + 149.0`
- Surface reflectance: `DN × 0.0000275 − 0.2`
- Cloud, cirrus, shadow and dilated-cloud masking from `qa_pixel` bit flags, applied
  **before** compositing — one undetected cloud drags a median down several degrees.

In [ ]:
LS_OPTICAL = ["green", "red", "nir08", "swir16"]
LS_ASSETS  = ["lwir11"] + LS_OPTICAL + ["qa_pixel"]
QA_BITS    = [0, 1, 2, 3, 4]        # fill, dilated cloud, cirrus, cloud, shadow

def landsat_epoch(bbox, window, label):
    """Cloud-masked summer median: LST in Celsius plus surface-reflectance bands."""
    items = list(catalog.search(
        collections=["landsat-c2-l2"], bbox=bbox, datetime=window,
        query={"eo:cloud_cover": {"lt": MAX_CLOUD},
               "platform": {"in": ["landsat-8", "landsat-9"]}}).items())
    if not items:
        raise RuntimeError(f"No Landsat scenes for {label}.")

    # Keep the cleanest N scenes rather than an arbitrary slice.
    items = sorted(items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
    items = items[:MAX_SCENES_LS]
    dates = sorted(i.properties["datetime"][:10] for i in items)
    print(f"{label}: using {len(items)} cleanest scenes, {dates[0]} -> {dates[-1]} "
          f"(cloud {items[0].properties['eo:cloud_cover']:.1f}-"
          f"{items[-1].properties['eo:cloud_cover']:.1f}%)")

    stack = stackstac.stack(items, assets=LS_ASSETS, resolution=RES_LST,
                            bounds_latlon=bbox, epsg=EPSG, rescale=False,
                            dtype="float32", fill_value=np.nan, chunksize=1024)

    qa = stack.sel(band="qa_pixel").astype("uint16")
    bad = sum(((qa >> b) & 1) for b in QA_BITS)

    st = stack.sel(band="lwir11")
    lst = ((st.where(st > 0) * 0.00341802 + 149.0) - 273.15).where(bad == 0)
    lst = lst.median(dim="time", skipna=True).compute()
    lst.name = f"LST_{label}"

    sr = stack.sel(band=LS_OPTICAL)
    sr = (sr * 0.0000275 - 0.2).where(bad == 0)
    sr = sr.median(dim="time", skipna=True).compute()

    print(f"   LST {np.nanmin(lst.values):.1f} to {np.nanmax(lst.values):.1f} C, "
          f"{int(np.isfinite(lst.values).sum()):,} valid pixels")
    return lst, sr, items

lst_a, sr_a, items_a = landsat_epoch(AOI_BBOX, EPOCH_A, EPOCH_A_NAME)
lst_b, sr_b, items_b = landsat_epoch(AOI_BBOX, EPOCH_B, EPOCH_B_NAME)
d_lst = (lst_b - lst_a).rename("delta_LST")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, arr, t in zip(axes[:2], [lst_a, lst_b],
                      [f"Summer LST {EPOCH_A_NAME}", f"Summer LST {EPOCH_B_NAME}"]):
    im = ax.imshow(arr.values, cmap="inferno", vmin=35, vmax=60)
    ax.set_title(t); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, label="degrees C")

lim = float(np.nanpercentile(np.abs(d_lst.values), 98))
im = axes[2].imshow(d_lst.values, cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[2].set_title(f"Change in LST, {EPOCH_A_NAME} to {EPOCH_B_NAME}"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046, label="degrees C")
plt.suptitle(f"Summer median land surface temperature - {AOI_NAME}",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{RESULTS}/01_lst_change.png", bbox_inches="tight")
plt.show()

print(f"Mean LST {EPOCH_A_NAME}: {np.nanmean(lst_a.values):.2f} C")
print(f"Mean LST {EPOCH_B_NAME}: {np.nanmean(lst_b.values):.2f} C")
print(f"Mean change            : {np.nanmean(d_lst.values):+.2f} C")

## 4 · Built-up classification and independent validation

Simple, auditable index thresholds — the programme guidance is explicit that a
simple explainable baseline comes before anything complex, and a municipal GIS
officer must be able to check the rule by hand.

Then the part most teams skip: check it against **ESA WorldCover 2021**, an
independently produced 10 m product, and report a full confusion matrix. This is
what the organizers mean by *"not just 'it looks right on the map'"*.

In [ ]:
eps = 1e-6
def indices(sr):
    g, r = sr.sel(band="green"), sr.sel(band="red")
    n, s = sr.sel(band="nir08"), sr.sel(band="swir16")
    return (dict(ndvi=((n - r) / (n + r + eps)).values,
                 ndbi=((s - n) / (s + n + eps)).values,
                 mndwi=((g - s) / (g + s + eps)).values))

def classify(ix):
    """0 water, 1 vegetation, 2 built-up, 3 bare desert. Order matters."""
    lc = np.full(ix["ndvi"].shape, 3, dtype=np.uint8)
    lc[ix["ndbi"] > 0.0] = 2
    lc[ix["ndvi"] > 0.20] = 1
    lc[ix["mndwi"] > 0.10] = 0
    return lc

ix_a, ix_b = indices(sr_a), indices(sr_b)
lc_a, lc_b = classify(ix_a), classify(ix_b)
LABELS = ["Water", "Vegetation", "Built-up", "Bare / desert"]

px_km2 = RES_LST ** 2 / 1e6
print(f"Land cover, {EPOCH_B_NAME}:")
for i, name in enumerate(LABELS):
    n = int((lc_b == i).sum())
    print(f"   {name:16s} {n:8,} px   {n*px_km2:7.2f} km2")

In [ ]:
# ---- Reference: ESA WorldCover 2021 (v200) ---------------------------------
wc_items = list(catalog.search(collections=["esa-worldcover"], bbox=AOI_BBOX).items())
wc_2021 = [i for i in wc_items if "2021" in i.id] or wc_items
print(f"Using WorldCover reference: {[i.id for i in wc_2021]}")

wc = stackstac.stack(wc_2021, assets=["map"], resolution=RES_LST,
                     bounds_latlon=AOI_BBOX, epsg=EPSG, rescale=False,
                     dtype="float32", fill_value=np.nan).isel(band=0)
wc = (wc.max(dim="time") if "time" in wc.dims else wc).compute()
wc_grid = wc.interp_like(lst_b, method="nearest").values

WC_BUILTUP = 50
ref   = (wc_grid == WC_BUILTUP)
ours  = (lc_b == 2)
valid = np.isfinite(wc_grid) & np.isfinite(lst_b.values)
ref, ours = ref[valid], ours[valid]

TP = int((ours & ref).sum());  FP = int((ours & ~ref).sum())
FN = int((~ours & ref).sum()); TN = int((~ours & ~ref).sum())
acc  = (TP+TN)/(TP+TN+FP+FN)
prec = TP/(TP+FP) if TP+FP else float("nan")
rec  = TP/(TP+FN) if TP+FN else float("nan")
f1   = 2*prec*rec/(prec+rec) if prec+rec else float("nan")
iou  = TP/(TP+FP+FN) if TP+FP+FN else float("nan")

print(f"\nBuilt-up classification vs ESA WorldCover 2021")
print(f"   pixels compared : {valid.sum():,}")
print(f"   TP={TP:,}  FP={FP:,}  FN={FN:,}  TN={TN:,}")
print(f"   accuracy  {acc:.3f}   precision {prec:.3f}   recall {rec:.3f}")
print(f"   F1        {f1:.3f}   IoU       {iou:.3f}")

validation_builtup = dict(n=int(valid.sum()), TP=TP, FP=FP, FN=FN, TN=TN,
                          accuracy=acc, precision=prec, recall=rec, f1=f1, iou=iou)
print("\nNOTE for README: WorldCover is itself a model output, not ground truth. "
      "This measures agreement with an independent published product.")

## 5 · Urban expansion and its thermal cost

The headline result. We compare the temperature change of land that was converted
from desert to built-up against land that stayed desert. Both experienced the same
regional climate over the same decade, so the difference **isolates the
urbanisation signal from the background warming trend**.

In [ ]:
new_builtup    = (lc_a == 3) & (lc_b == 2)
stable_builtup = (lc_a == 2) & (lc_b == 2)
stable_desert  = (lc_a == 3) & (lc_b == 3)
dl = d_lst.values

rows = []
for name, m in [("New built-up (was desert)", new_builtup),
                ("Stable built-up", stable_builtup),
                ("Stable desert", stable_desert)]:
    mm = m & np.isfinite(dl)
    if mm.sum() < 30:
        continue
    rows.append(dict(category=name, n_pixels=int(mm.sum()),
                     area_km2=round(float(mm.sum()*px_km2), 2),
                     LST_before=round(float(np.nanmean(lst_a.values[mm])), 2),
                     LST_after=round(float(np.nanmean(lst_b.values[mm])), 2),
                     delta_LST=round(float(np.nanmean(dl[mm])), 2)))

transition = pd.DataFrame(rows)
transition.to_csv(f"{RESULTS}/transition_summary.csv", index=False)
display(transition)

HEADLINE = None
try:
    conv = transition.loc[transition.category.str.startswith("New"), "delta_LST"].iloc[0]
    des  = transition.loc[transition.category.str.startswith("Stable desert"), "delta_LST"].iloc[0]
    HEADLINE = round(float(conv - des), 2)
    print(f"\nHEADLINE: land converted from desert to built-up warmed "
          f"{HEADLINE:+.2f} C MORE than desert left untouched.")
    print(f"Converted area: {float(new_builtup.sum()*px_km2):.2f} km2 "
          f"({float(new_builtup.sum()*px_km2*100):,.0f} hectares)")
except (IndexError, KeyError):
    print("Could not compute the headline contrast - check class counts above.")

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
chg = np.zeros_like(lc_b, dtype=np.uint8)
chg[stable_desert], chg[stable_builtup], chg[new_builtup] = 1, 2, 3
axes[0].imshow(chg, cmap=ListedColormap(["#ffffff","#f0e0c0","#9e9e9e","#d32f2f"]),
               vmin=0, vmax=3); axes[0].axis("off")
axes[0].set_title(f"Urban expansion {EPOCH_A_NAME} to {EPOCH_B_NAME}")
axes[0].legend(handles=[Patch(color="#f0e0c0", label="Desert (unchanged)"),
                        Patch(color="#9e9e9e", label="Built-up (unchanged)"),
                        Patch(color="#d32f2f", label="Newly built-up")],
               loc="lower right", fontsize=8, framealpha=0.9)

axes[1].boxplot([dl[m & np.isfinite(dl)] for m in
                 (stable_desert, stable_builtup, new_builtup)],
                labels=["Stable\ndesert","Stable\nbuilt-up","Newly\nbuilt-up"],
                showfliers=False)
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("Change in summer LST (degrees C)")
axes[1].set_title("Temperature change by land-use transition")
axes[1].grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(f"{RESULTS}/03_expansion_heat.png", bbox_inches="tight")
plt.show()

## 6 · Broadband albedo from Sentinel-2

Albedo is the physical property a cool roof changes, so it must be measured, not
assumed. Sentinel-2 is used here only — present day, no cross-epoch comparison —
where its 20 m resolution resolves individual large roofs that Landsat's 30 m
thermal pixel cannot.

Narrow-to-broadband conversion:

```
α = 0.2266·B2 + 0.1236·B3 + 0.1573·B4 + 0.3417·B8 + 0.1170·B11 + 0.0338·B12
```

Weights sum to 1.0.

> **Meera / team — before submission:** confirm and cite the coefficient source
> (Bonafoni & Sekertekin, *Albedo Retrieval From Sentinel-2 by New Narrow-to-Broadband
> Conversion Coefficients*, IEEE GRSL 2020) in README §5. Do not ship an uncited
> constant — *validity* is judged, and an unsourced formula is the first thing a
> reviewer questions.

In [ ]:
S2_BANDS = ["B02","B03","B04","B08","B11","B12"]
ALB_W    = dict(B02=0.2266, B03=0.1236, B04=0.1573, B08=0.3417, B11=0.1170, B12=0.0338)
SCL_BAD  = [0,1,3,8,9,10,11]      # nodata, saturated, shadow, clouds, cirrus, snow

items_s2 = list(catalog.search(collections=["sentinel-2-l2a"], bbox=AOI_BBOX,
                               datetime=S2_WINDOW,
                               query={"eo:cloud_cover": {"lt": MAX_CLOUD}}).items())
items_s2 = sorted(items_s2, key=lambda i: i.properties.get("eo:cloud_cover", 100))[:MAX_SCENES_S2]
print(f"Sentinel-2: using {len(items_s2)} cleanest scenes from {S2_WINDOW}")

s2 = stackstac.stack(items_s2, assets=S2_BANDS + ["SCL"], resolution=RES_S2,
                     bounds_latlon=AOI_BBOX, epsg=EPSG, rescale=False,
                     dtype="float32", fill_value=np.nan, chunksize=1024)
good = ~s2.sel(band="SCL").isin(SCL_BAD)
refl = (s2.sel(band=S2_BANDS).where(good) / 10000.0).median(dim="time", skipna=True).compute()

albedo = sum(w * refl.sel(band=n) for n, w in ALB_W.items()).rename("albedo")
albedo = albedo.rio.write_crs(EPSG)
print(f"albedo p1={np.nanpercentile(albedo.values,1):.3f}  "
      f"median={np.nanmedian(albedo.values):.3f}  "
      f"p99={np.nanpercentile(albedo.values,99):.3f}")

plt.figure(figsize=(8, 6.5))
plt.imshow(albedo.values, cmap="gray", vmin=0.05, vmax=0.55)
plt.colorbar(label="Broadband albedo", fraction=0.046)
plt.title(f"Surface albedo - {AOI_NAME}, {S2_WINDOW[:4]}"); plt.axis("off")
plt.savefig(f"{RESULTS}/02_albedo.png", bbox_inches="tight"); plt.show()

## 7 · Calibrating the lever — how much cooling does albedo buy *here*?

The analytical core.

Over built-up pixels only, we regress summer LST on albedo, controlling for
vegetation. The slope is the **locally measured sensitivity of surface temperature
to albedo** — °C per unit albedo, derived from this city rather than imported from a
study of somewhere else. This is what converts a heat map into a decision tool.

**Validation is spatial, not random.** Neighbouring pixels are strongly
autocorrelated; a random train/test split leaks near-identical pixels across the
boundary and reports a flattering, meaningless R². The programme guidance flags this
exact mistake. We hold out whole ~1 km blocks.

In [ ]:
# Put LST and land cover on the Sentinel-2 (20 m) grid to match albedo.
lst_20 = lst_b.rio.write_crs(EPSG).interp_like(albedo, method="linear").values
lc_20  = xr.DataArray(lc_b, dims=lst_b.dims, coords=lst_b.coords
                      ).rio.write_crs(EPSG).interp_like(albedo, method="nearest").values
alb    = albedo.values

built = (lc_20 == 2) & np.isfinite(lst_20) & np.isfinite(alb)
ndvi_20 = xr.DataArray(ix_b["ndvi"], dims=lst_b.dims, coords=lst_b.coords
                       ).rio.write_crs(EPSG).interp_like(albedo, method="linear").values
built &= np.isfinite(ndvi_20)
print(f"Built-up pixels for the model: {built.sum():,}")

y = lst_20[built]
X = np.column_stack([alb[built], ndvi_20[built]])

# ---- spatial block cross-validation ----------------------------------------
BLOCK = int(1000 / RES_S2)                       # ~1 km blocks
rr, cc = np.where(built)
bid = (rr // BLOCK) * 10_000 + (cc // BLOCK)
blocks = np.unique(bid); np.random.shuffle(blocks)
K = 5
fold = np.array([{b: i % K for i, b in enumerate(blocks)}[v] for v in bid])
print(f"{len(blocks)} spatial blocks, {K}-fold block CV")

oof, slopes = np.full_like(y, np.nan, dtype=float), []
for k in range(K):
    tr, te = fold != k, fold == k
    if te.sum() < 30 or tr.sum() < 100:
        continue
    m = LinearRegression().fit(X[tr], y[tr])
    oof[te] = m.predict(X[te]); slopes.append(m.coef_[0])

ok = np.isfinite(oof)
cv_r2   = r2_score(y[ok], oof[ok])
cv_rmse = float(np.sqrt(mean_squared_error(y[ok], oof[ok])))
final   = LinearRegression().fit(X, y)
SENS    = float(final.coef_[0])

print(f"\n--- Albedo -> LST model, built-up pixels only ---")
print(f"  LST = {final.intercept_:.2f} {final.coef_[0]:+.2f}*albedo {final.coef_[1]:+.2f}*NDVI")
print(f"  in-sample R2    : {final.score(X, y):.3f}")
print(f"  spatial-CV R2   : {cv_r2:.3f}")
print(f"  spatial-CV RMSE : {cv_rmse:.2f} C")
if slopes:
    print(f"  slope across folds: {min(slopes):.2f} to {max(slopes):.2f} (sd {np.std(slopes):.2f})")
print(f"\n  MEASURED SENSITIVITY: {SENS:.1f} C per unit albedo")
print(f"  -> +0.10 albedo lowers summer surface temperature by ~{abs(SENS)*0.10:.2f} C")

validation_model = dict(cv_r2=cv_r2, cv_rmse=cv_rmse, n=int(built.sum()),
                        n_blocks=int(len(blocks)), folds=K, slope=SENS,
                        slope_sd=float(np.std(slopes)) if slopes else None)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
idx = np.random.choice(len(y), size=min(20000, len(y)), replace=False)
axes[0].scatter(X[idx,0], y[idx], s=1.5, alpha=0.12, c="#1f3a5f")
xs = np.linspace(np.percentile(X[:,0],1), np.percentile(X[:,0],99), 50)
axes[0].plot(xs, final.intercept_ + final.coef_[0]*xs + final.coef_[1]*np.median(X[:,1]),
             c="#d32f2f", lw=2.2, label=f"{final.coef_[0]:.1f} C per unit albedo")
axes[0].set_xlabel("Broadband albedo"); axes[0].set_ylabel("Summer LST (degrees C)")
axes[0].set_title("Darker surfaces run hotter - measured locally")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(oof[ok][:20000], y[ok][:20000], s=1.5, alpha=0.12, c="#1f3a5f")
lo, hi = float(np.nanmin(oof[ok])), float(np.nanmax(oof[ok]))
axes[1].plot([lo,hi],[lo,hi],"k--",lw=1)
axes[1].set_xlabel("Predicted LST, held-out blocks"); axes[1].set_ylabel("Observed LST")
axes[1].set_title(f"Spatial block CV: R2={cv_r2:.2f}, RMSE={cv_rmse:.2f} C")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS}/04_albedo_model.png", bbox_inches="tight")
plt.show()

## 8 · From pixels to a worklist — ranking individual rooftops

```
ΔT      = |sensitivity| × max(0, 0.60 − current_albedo)
benefit = ΔT × roof_area_m²        [°C·m²]
```

Area is in the formula deliberately: one large warehouse roof is far cheaper to treat
per square metre than fifty small ones, so the first works contract should cover the
biggest, darkest roofs.

In [ ]:
def osm_buildings(bbox, timeout=300):
    """Overpass directly - one HTTP call, no library API churn across versions."""
    w, s, e, n = bbox
    q = f'[out:json][timeout:240];(way["building"]({s},{w},{n},{e});' \
        f'relation["building"]({s},{w},{n},{e}););out geom;'
    for url in ["https://overpass-api.de/api/interpreter",
                "https://overpass.kumi.systems/api/interpreter"]:
        try:
            r = requests.post(url, data={"data": q}, timeout=timeout)
            r.raise_for_status(); data = r.json(); break
        except Exception as ex:
            print(f"   {url.split('/')[2]} failed ({type(ex).__name__}), next mirror")
    else:
        raise RuntimeError("All Overpass mirrors failed - retry in a few minutes.")

    polys, props = [], []
    for el in data.get("elements", []):
        g = el.get("geometry")
        if not g or len(g) < 4:
            continue
        try:
            p = Polygon([(d["lon"], d["lat"]) for d in g])
            if p.is_valid and p.area > 0:
                polys.append(p)
                props.append({"osm_id": el.get("id"),
                              **{k: v for k, v in (el.get("tags") or {}).items()
                                 if k in ("building", "name")}})
        except Exception:
            continue
    return gpd.GeoDataFrame(props, geometry=polys, crs="EPSG:4326")

print("Querying OpenStreetMap (1-3 minutes)...")
roofs = osm_buildings(FOCUS_BBOX).to_crs(epsg=EPSG)
roofs["area_m2"] = roofs.geometry.area
roofs = roofs[roofs.area_m2 >= 400].reset_index(drop=True)
print(f"{len(roofs):,} roofs >= 400 m2, total {roofs.area_m2.sum()/1e6:.2f} km2 of roof")

In [ ]:
affine = albedo.rio.transform()
def zmean(arr):
    st = zonal_stats(roofs, arr.astype("float32"), affine=affine,
                     stats=["mean"], nodata=np.nan)
    return np.array([s["mean"] if s and s["mean"] is not None else np.nan for s in st])

print("Zonal statistics per roof...")
roofs["albedo"] = zmean(alb)
roofs["lst_c"]  = zmean(lst_20)
roofs = roofs.dropna(subset=["albedo", "lst_c"]).reset_index(drop=True)
print(f"{len(roofs):,} roofs with valid albedo and LST")

s = abs(SENS)
roofs["albedo_gap"]    = (TARGET_ALBEDO - roofs.albedo).clip(lower=0)
roofs["delta_T_est_c"] = s * roofs.albedo_gap
roofs["cooling_c_m2"]  = roofs.delta_T_est_c * roofs.area_m2
roofs["lst_anomaly_c"] = roofs.lst_c - roofs.lst_c.median()
roofs = roofs.sort_values("cooling_c_m2", ascending=False).reset_index(drop=True)
roofs["rank"] = np.arange(1, len(roofs) + 1)

cols = ["rank","osm_id","area_m2","albedo","lst_c","lst_anomaly_c",
        "albedo_gap","delta_T_est_c","cooling_c_m2"]
out = roofs[cols].round({"area_m2":0,"albedo":3,"lst_c":2,"lst_anomaly_c":2,
                         "albedo_gap":3,"delta_T_est_c":2,"cooling_c_m2":0})
out.to_csv(f"{RESULTS}/roof_priority_ranked.csv", index=False)
roofs.head(500)[["rank","osm_id","area_m2","albedo","lst_c",
                 "delta_T_est_c","cooling_c_m2","geometry"]].to_file(
    f"{RESULTS}/top500_roofs.geojson", driver="GeoJSON")

top = roofs.head(100)
print(f"\nTop 100 roofs: {top.area_m2.sum()/1e4:,.1f} ha "
      f"({top.area_m2.sum()/roofs.area_m2.sum()*100:.1f}% of roof area), "
      f"carrying {top.cooling_c_m2.sum()/roofs.cooling_c_m2.sum()*100:.1f}% of total benefit")
print(f"  mean albedo {top.albedo.mean():.3f}, mean LST {top.lst_c.mean():.1f} C, "
      f"mean estimated cooling {top.delta_T_est_c.mean():.2f} C")
display(out.head(15))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
roofs.plot(ax=axes[0], column="lst_c", cmap="inferno", legend=True,
           legend_kwds={"label":"Mean roof LST (C)","shrink":0.7})
axes[0].set_title("Every roof, by surface temperature"); axes[0].set_axis_off()

roofs.plot(ax=axes[1], color="#e8e8e8"); top.plot(ax=axes[1], color="#d32f2f")
axes[1].set_title("Top 100 priority roofs (red)"); axes[1].set_axis_off()

axes[2].scatter(roofs.albedo, roofs.lst_c, s=np.clip(roofs.area_m2/900, 2, 60),
                alpha=0.35, c="#1f3a5f", edgecolors="none")
axes[2].set_xlabel("Roof albedo"); axes[2].set_ylabel("Roof LST (degrees C)")
axes[2].set_title("Roofs: albedo vs temperature\n(marker size = roof area)")
axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{RESULTS}/05_roof_priority.png", bbox_inches="tight")
plt.show()

## 9 · Export everything

In [ ]:
summary = {
    "aoi": AOI_NAME, "aoi_bbox": AOI_BBOX, "focus_bbox": FOCUS_BBOX,
    "epochs": {"before": EPOCH_A_NAME, "after": EPOCH_B_NAME},
    "scenes_used": {"landsat_before": len(items_a), "landsat_after": len(items_b),
                    "sentinel2": len(items_s2)},
    "mean_lst_before_c": round(float(np.nanmean(lst_a.values)), 2),
    "mean_lst_after_c":  round(float(np.nanmean(lst_b.values)), 2),
    "desert_to_builtup_km2": round(float(new_builtup.sum()*px_km2), 2),
    "headline_extra_warming_c": HEADLINE,
    "albedo_sensitivity_c_per_unit": round(SENS, 2),
    "cooling_per_0.1_albedo_c": round(abs(SENS)*0.1, 2),
    "validation_builtup": {k: (round(v,4) if isinstance(v,float) else v)
                           for k,v in validation_builtup.items()},
    "validation_model":   {k: (round(v,4) if isinstance(v,float) else v)
                           for k,v in validation_model.items()},
    "roofs_analysed": int(len(roofs)),
    "roof_area_total_km2": round(float(roofs.area_m2.sum()/1e6), 3),
    "top100_benefit_share_pct": round(float(
        roofs.head(100).cooling_c_m2.sum()/roofs.cooling_c_m2.sum()*100), 1),
}
with open(f"{RESULTS}/summary.json","w") as f:
    json.dump(summary, f, indent=2)

try:
    bnds = gpd.GeoSeries([box(*FOCUS_BBOX)], crs=4326).to_crs(EPSG).total_bounds
    albedo.rio.clip_box(*bnds).rio.to_raster(f"{SAMPLE}/albedo_focus.tif", compress="deflate")
except Exception as e:
    print("sample clip failed:", e)

provenance = {
    "generated_utc": pd.Timestamp.utcnow().isoformat(),
    "aoi_bbox": AOI_BBOX, "focus_bbox": FOCUS_BBOX, "epsg": EPSG,
    "landsat": {"collection":"landsat-c2-l2", "windows":[EPOCH_A, EPOCH_B],
                "max_cloud_pct":MAX_CLOUD, "max_scenes":MAX_SCENES_LS,
                "scene_ids_before":[i.id for i in items_a],
                "scene_ids_after":[i.id for i in items_b]},
    "sentinel2": {"collection":"sentinel-2-l2a","window":S2_WINDOW,
                  "bands":S2_BANDS,"scene_ids":[i.id for i in items_s2]},
    "worldcover": {"items":[i.id for i in wc_2021],"builtup_class":50},
    "osm": {"source":"Overpass API","bbox":FOCUS_BBOX,"min_roof_area_m2":400},
    "licences": {"Landsat":"Public domain (USGS)",
                 "Sentinel-2":"Copernicus Sentinel data, free and open",
                 "ESA WorldCover":"CC BY 4.0",
                 "OpenStreetMap":"ODbL (c) OpenStreetMap contributors"},
}
with open(f"{SAMPLE}/provenance.json","w") as f:
    json.dump(provenance, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nFiles written:")
for d in (RESULTS, SAMPLE):
    for fn in sorted(os.listdir(d)):
        print(f"   {d}/{fn:38s} {os.path.getsize(os.path.join(d,fn))/1e3:9.1f} KB")

---

## 10 · Limitations — state these plainly, they are scored under *validity*

- **LST is not air temperature.** Satellites measure the radiating skin of the
  surface. Human thermal comfort depends on air temperature, humidity and radiant
  load. This ranks *surfaces*; it does not predict heat-stroke risk.
- **The sensitivity is correlational.** §7 measures how albedo and LST covary across
  space. It is not a controlled experiment, and confounders — roof thermal mass,
  building use, HVAC exhaust, roof age — are not separated. The ΔT figures are
  screening estimates with real uncertainty, not engineering predictions.
- **Validation is against a model, not ground truth.** ESA WorldCover is itself a
  classification product. No field survey or in-situ logging was performed.
- **Mixed thermal pixels.** Landsat thermal is 30 m; roofs below ~900 m² carry signal
  from surroundings. Roofs under 400 m² are excluded; the large-roof end of the
  ranking is the trustworthy end.
- **OSM completeness varies.** Missing footprints are missing from the ranking. A
  deployed version would use municipal cadastral data.
- **Overpass time.** Landsat crosses at ~10:30 local solar time; peak surface
  temperature is later in the afternoon, so absolute values understate the daily
  maximum. Relative ranking between roofs is unaffected.
- **Hyperspectral.** No Planet Tanager `urban` scene covers the UAE (verified in
  `00_recon.ipynb`). See the separate Tier 2 notebook for how we handled this.

## How this reaches a decision-maker

Not built for the PoC, but the path is:

1. **Quarterly screening report.** The pipeline reruns each summer and delivers a
   refreshed top-200 worklist as CSV/PDF — no new software needed on their side.
2. **GIS layer.** `top500_roofs.geojson` drops straight into the municipality's
   existing ArcGIS/QGIS environment alongside cadastral and permit layers.
3. **API.** A `/priority?district=` endpoint for works-planning systems.
4. **Verification loop.** After a retrofit, rerun on the next summer's imagery and
   compare realised ΔT against predicted — turning the tool into a contract
   compliance check, not just a planning aid.
